In [0]:
%pip uninstall -y psycopg2-binary
%pip install -q sentence-transformers pandas sqlalchemy
# | Package               | Purpose                                     |
# | --------------------- | ------------------------------------------- |
# | sentence-transformers | Convert weather text → 384 dimension vector |
# | pandas                | Easy data handling                          |
# | psycopg2              | Connect and write to Lakebase PostgreSQL    |



In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd

from sentence_transformers import SentenceTransformer

from lakebase import get_connection

print("Imports successful")

In [0]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

CHUNK_SIZE = 800
CHUNK_OVERLAP = 100

DOCUMENT_TABLE = "weather_documents"
EMBEDDING_TABLE = "weather_chunk_embeddings"

print("Configuration loaded")

In [0]:
print("Loading model...")

model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print("Model loaded")

In [0]:
# Test model

text = "Heavy rainfall expected causing flooding near rivers"

vector = model.encode(text)

print(type(vector))
print("Dimensions:", len(vector))

In [0]:
#Query weather Documents
query = """
SELECT
    id,
    location,
    source_type,
    headline,
    narrative_text
FROM weather_documents
WHERE narrative_text IS NOT NULL
AND narrative_text <> ''
LIMIT 10;
"""


with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute(query)
        rows = cur.fetchall()


print("Documents loaded:", len(rows))
rows[:2]

In [0]:
# Convert to DataFrame

# Embeddings are easier with pandas.


documents_df = pd.DataFrame(rows)

documents_df.head()

In [0]:

# Create embedding text column

# Location 
# +
# headline
# +
# narrative_text



documents_df["embedding_text"] = (
    "Weather information for "
    + documents_df["location"].fillna("")
    + ". "
    + documents_df["headline"].fillna("")
    + ". "
    + documents_df["narrative_text"].fillna("")
)

In [0]:
documents_df

In [0]:
def chunk_text(text, chunk_size=800, overlap=100):
    """
    Split text into overlapping chunks.
    """

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [0]:
#Test chunk code
sample_text = documents_df.iloc[0]["embedding_text"]

chunks = chunk_text(sample_text)

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print("\nChunk", i)
    print(chunk[:200])

In [0]:
#Apply chunk to all documents

In [0]:
#Apply chunk to all documents
chunk_rows = []

for _, row in documents_df.iterrows():

    chunks = chunk_text(
        row["embedding_text"]
    )

    for index, chunk in enumerate(chunks):

        chunk_rows.append(
            {
                "document_id": row["id"],
                "location": row["location"],
                "chunk_index": index,
                "chunk_text": chunk
            }
        )


chunks_df = pd.DataFrame(chunk_rows)

print("Total chunks:", len(chunks_df))

chunks_df.head()

In [0]:
# Generate embeddings for every chunk.
# Load the embedding model
from sentence_transformers import SentenceTransformer
import os


MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"


# HuggingFace cache location
os.environ["HF_HOME"] = "/tmp/.cache/huggingface"


print("Loading model...")

model = SentenceTransformer(
    MODEL_NAME,
    cache_folder="/tmp/.cache/huggingface"
)


print("Model loaded successfully")

In [0]:
#Test with one chunk


sample = chunks_df.iloc[0]["chunk_text"]


print(sample)

In [0]:
vector = model.encode(sample)


print(type(vector))

print(vector.shape)

In [0]:
# Generate embeddings for ALL chunks

chunk_texts = chunks_df["chunk_text"].tolist()


print("Total chunks:", len(chunk_texts))

In [0]:
embeddings = model.encode(
    chunk_texts,
    show_progress_bar=True,
    batch_size=32
)

In [0]:
# Attach vectors back to dataframe
chunks_df["embedding"] = embeddings.tolist()
chunks_df.head()

In [0]:
# Verify one vector

first_vector = chunks_df.iloc[0]["embedding"]

print(len(first_vector))

print(first_vector[:5])

In [0]:
chunks_df.shape

In [0]:
#Get lakebase connection details
import base64
from urllib.parse import urlparse
from databricks.sdk import WorkspaceClient


w = WorkspaceClient()


def get_lakebase_url():

    secret = w.secrets.get_secret(
        scope="database",
        key="support-lakebase-url"
    )

    return base64.b64decode(
        secret.value
    ).decode("utf-8")


lakebase_url = get_lakebase_url()


parsed = urlparse(lakebase_url)


db_host = parsed.hostname
db_port = parsed.port or 5432
db_name = parsed.path.lstrip("/")
db_user = parsed.username
db_password = parsed.password


print("Database:", db_name)
print("Host:", db_host)

In [0]:
#connect to database
import psycopg2


conn = psycopg2.connect(
    host=db_host,
    port=db_port,
    dbname=db_name,
    user=db_user,
    password=db_password,
    sslmode="require"
)


cursor = conn.cursor()


print("Connected to Lakebase")

In [0]:
# Check your dataframe first

chunks_df.columns

In [0]:
#create insert row
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"


insert_rows = []


for _, row in chunks_df.iterrows():

    chunk_id = f"{row['document_id']}_{row['chunk_index']}"


    insert_rows.append(
        (
            chunk_id,
            row["document_id"],
            int(row["chunk_index"]),
            row["chunk_text"],
            str(row["embedding"]),
            MODEL_NAME
        )
    )


print("Total rows prepared:", len(insert_rows))

In [0]:
conn.rollback()

print("Transaction reset")

In [0]:
cursor.execute("SELECT 1;")

print(cursor.fetchone())

In [0]:
cursor.execute("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema='public'
ORDER BY table_name;
""")

tables = cursor.fetchall()

for t in tables:
    print(t[0])

In [0]:
cursor.execute(
    "SELECT COUNT(*) FROM weather_embeddings;"
)

print(cursor.fetchone())

In [0]:
cursor.close()
conn.close()


conn = psycopg2.connect(
    host=db_host,
    port=db_port,
    dbname=db_name,
    user=db_user,
    password=db_password,
    sslmode="require"
)

cursor = conn.cursor()

print("Fresh connection")

In [0]:
#Insert embeddings into weather_embeddings

insert_sql = """

INSERT INTO weather_embeddings
(
    id,
    document_id,
    chunk_index,
    chunk_text,
    embedding,
    model_name
)

VALUES
(
    %s,
    %s,
    %s,
    %s,
    %s::vector,
    %s
)

ON CONFLICT (id)
DO NOTHING;

"""


cursor.executemany(
    insert_sql,
    insert_rows
)


conn.commit()


print(
    "Inserted rows:",
    cursor.rowcount
)

In [0]:
cursor.execute(
    "SELECT COUNT(*) FROM weather_embeddings;"
)

print(cursor.fetchone())

In [0]:
# stored chunks + vector
cursor.execute("""
SELECT
    id,
    document_id,
    chunk_index,
    LEFT(chunk_text,100),
    vector_dims(embedding)
FROM weather_embeddings
LIMIT 5;
""")


for row in cursor.fetchall():
    print(row)

In [0]:
# %sql
# Create the query embedding.
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Model loaded")

In [0]:
# Give it a test question
query = "Will there be flooding near rivers?"

print(query)

In [0]:
# Convert the question into a vector
query_embedding = model.encode(query)

print(type(query_embedding))
print(len(query_embedding))

In [0]:
# convert numpy vector into PostgreSQL format
query_vector = (
    "[" +
    ",".join(
        map(str, query_embedding.tolist())
    )
    + "]"
)

print(query_vector[:100])

In [0]:
# Close old objects safely

try:
    cursor.close()
    conn.close()
except:
    pass

print("Old connection closed")

In [0]:
#EXECUTE lakebase connection steps (23 and 24 cells as now)

In [0]:
cursor.execute("SELECT 1;")

print(cursor.fetchone())

In [0]:
#Run similarity Search
sql = """
SELECT
    document_id,
    chunk_index,
    chunk_text,

    1 - (embedding <=> %s::vector) AS similarity

FROM weather_embeddings

ORDER BY embedding <=> %s::vector

LIMIT 5;
"""


cursor.execute(
    sql,
    (
        query_vector,
        query_vector
    )
)


results = cursor.fetchall()


for row in results:
    print("--------------------------------")
    print("Document:", row[0])
    print("Chunk:", row[1])
    print("Similarity:", round(row[3], 4))
    print(row[2][:200])